In [57]:
# import necessary libraries
import pandas as pd
import numpy as np

In [58]:
# Load the cleaned datasets
crime_df = pd.read_csv("Datasets/Processed-Data/Police-Data/Street_Police_Data_Cleaned_LSAO_2020_2024.csv")
income_df = pd.read_csv("Datasets/Processed-Data/Income-Data/Income_Cleaned_MSOA_2020_2024.csv")
population_df = pd.read_csv("Datasets/Processed-Data/Population-Data/Population_Cleaned_LSOA_2020_2024.csv")
weather_df = pd.read_csv("Datasets/Processed-Data/Weather-Data/Heathrow_Monthly_Weather_Cleaned_1948_2025.csv")
lsoa_df = pd.read_csv("Datasets/Processed-Data/Map-Data/LSOA_Map_Cleaned.csv")
msoa_df = pd.read_csv("Datasets/Processed-Data/Map-Data/MSOA_Map_Cleaned.csv")
map_lookup_df = pd.read_csv("Datasets/Processed-Data/Map-Data/LSOA_MSOA_Lookup_Cleaned.csv")

## Task 1 - Dataset for Crime Per Capita, Total , and Crime Type Prediction

In [59]:
# Check for missing values in the crime dataset
crime_df.notnull().sum()

Crime_ID                 5564310
Longitude                5564310
Latitude                 5564310
Location                 5564310
LSOA_Code                5564310
LSOA_Name                5564310
Crime_Type               5564310
Last_Outcome_Category    5564310
Month_Period             5564310
Year                     5564310
Month                    5564310
dtype: int64

In [60]:
# Get unique LSOA codes from each dataframe
crime_lsoa_codes = set(crime_df['LSOA_Code'].unique())
lsoa_df_codes = set(map_lookup_df['LSOA_Code'].unique())

# Find codes that are in both
matching_codes = crime_lsoa_codes & lsoa_df_codes

# Summary
print(f"Total unique LSOA codes in crime_df: {len(crime_lsoa_codes)}")
print(f"Total unique LSOA codes in map_lookup_df: {len(lsoa_df_codes)}")
print(f"LSOA codes from crime_df present in map_lookup_df: {len(matching_codes)}")
print(f"\nMatching LSOA codes ({len(matching_codes)} total):")
print(sorted(matching_codes))

Total unique LSOA codes in crime_df: 12907
Total unique LSOA codes in map_lookup_df: 43916
LSOA codes from crime_df present in map_lookup_df: 12415

Matching LSOA codes (12415 total):
['E01000001', 'E01000002', 'E01000003', 'E01000005', 'E01000006', 'E01000007', 'E01000008', 'E01000009', 'E01000011', 'E01000012', 'E01000013', 'E01000014', 'E01000015', 'E01000016', 'E01000017', 'E01000018', 'E01000019', 'E01000020', 'E01000021', 'E01000022', 'E01000024', 'E01000025', 'E01000027', 'E01000028', 'E01000029', 'E01000030', 'E01000031', 'E01000032', 'E01000033', 'E01000034', 'E01000035', 'E01000036', 'E01000037', 'E01000038', 'E01000039', 'E01000040', 'E01000041', 'E01000042', 'E01000043', 'E01000044', 'E01000045', 'E01000046', 'E01000049', 'E01000050', 'E01000051', 'E01000052', 'E01000053', 'E01000054', 'E01000055', 'E01000056', 'E01000057', 'E01000058', 'E01000059', 'E01000060', 'E01000061', 'E01000062', 'E01000063', 'E01000064', 'E01000065', 'E01000066', 'E01000067', 'E01000068', 'E0100006

In [61]:
# Get the set of valid LSOA codes from lsoa_df
valid_lsoa_codes = set(map_lookup_df['LSOA_Code'].unique())

# Keep only rows where LSOA_Code is in lsoa_df
crime_df_filtered = crime_df[crime_df['LSOA_Code'].isin(valid_lsoa_codes)].copy()

# Show summary
print(f"Original crime_df shape: {crime_df.shape}")
print(f"Filtered crime_df shape: {crime_df_filtered.shape}")
print(f"Rows removed: {crime_df.shape[0] - crime_df_filtered.shape[0]}")

# If you want to update crime_df in place:
crime_df = crime_df_filtered

Original crime_df shape: (5564310, 11)
Filtered crime_df shape: (5231249, 11)
Rows removed: 333061


In [62]:
# Get unique LSOA codes from each dataframe
crime_lsoa_codes = set(crime_df['LSOA_Code'].unique())
population_df_codes = set(population_df['LSOA_Code'].unique())

# Find codes that are in both
matching_codes = crime_lsoa_codes & population_df_codes

# Summary
print(f"Total unique LSOA codes in crime_df: {len(crime_lsoa_codes)}")
print(f"Total unique LSOA codes in population_df: {len(lsoa_df_codes)}")
print(f"LSOA codes from crime_df present in population_df: {len(matching_codes)}")
print(f"\nMatching LSOA codes ({len(matching_codes)} total):")
print(sorted(matching_codes))

Total unique LSOA codes in crime_df: 12415
Total unique LSOA codes in population_df: 43916
LSOA codes from crime_df present in population_df: 12415

Matching LSOA codes (12415 total):
['E01000001', 'E01000002', 'E01000003', 'E01000005', 'E01000006', 'E01000007', 'E01000008', 'E01000009', 'E01000011', 'E01000012', 'E01000013', 'E01000014', 'E01000015', 'E01000016', 'E01000017', 'E01000018', 'E01000019', 'E01000020', 'E01000021', 'E01000022', 'E01000024', 'E01000025', 'E01000027', 'E01000028', 'E01000029', 'E01000030', 'E01000031', 'E01000032', 'E01000033', 'E01000034', 'E01000035', 'E01000036', 'E01000037', 'E01000038', 'E01000039', 'E01000040', 'E01000041', 'E01000042', 'E01000043', 'E01000044', 'E01000045', 'E01000046', 'E01000049', 'E01000050', 'E01000051', 'E01000052', 'E01000053', 'E01000054', 'E01000055', 'E01000056', 'E01000057', 'E01000058', 'E01000059', 'E01000060', 'E01000061', 'E01000062', 'E01000063', 'E01000064', 'E01000065', 'E01000066', 'E01000067', 'E01000068', 'E0100006

In [63]:
# Step 1: Aggregate crime counts by spatial-temporal groups
crime_agg = crime_df.groupby(['LSOA_Code', 'LSOA_Name', 'Year', 'Month', 'Month_Period']).size().reset_index(name='Crime_Count')

# Step 2: Merge with population data for normalization
crime_per_capita_df = crime_agg.merge(
    population_df[['LSOA_Code', 'Year', 'Total_Population']],
    on=['LSOA_Code', 'Year'],
    how='left'
)

# Step 3: Add LSOA spatial features (latitude, longitude, shape area, shape length)
# First, get unique LSOA features (handle duplicates by taking the first)
lsoa_features = lsoa_df.drop_duplicates(subset=['LSOA_Code'], keep='first')[
    ['LSOA_Code', 'Latitude', 'Longitude', 'Shape_Area', 'Shape_Length']
].rename(columns={
    'Latitude': 'LSOA_Latitude',
    'Longitude': 'LSOA_Longitude',
    'Shape_Area': 'LSOA_Shape_Area',
    'Shape_Length': 'LSOA_Shape_Length'
})

crime_per_capita_df = crime_per_capita_df.merge(
    lsoa_features,
    on='LSOA_Code',
    how='left'
)

# Step 4: Add MSOA hierarchical grouping
crime_per_capita_df = crime_per_capita_df.merge(
    map_lookup_df[['LSOA_Code', 'MSOA_Code']].drop_duplicates(),
    on='LSOA_Code',
    how='left'
)

# Step 5: Add income data via MSOA
crime_per_capita_df = crime_per_capita_df.merge(
    income_df[['MSOA_Code', 'MSOA_Name', 'Year', 'Total_Annual_Income_British_Pounds']],
    on=['MSOA_Code', 'Year'],
    how='left'
)

# Step 6: Add weather features (aligned by Year and Month)
crime_per_capita_df = crime_per_capita_df.merge(
    weather_df[['Year', 'Month', 'Max_Temperature_Celsius', 'Min_Temperature_Celsius', 
                 'Rainfall_mm', 'Sunshine_Hours', 'Air_Frost_Days']],
    on=['Year', 'Month'],
    how='left'
)

# Step 7: Calculate per-capita metric
crime_per_capita_df['Crime_per_Capita'] = (crime_per_capita_df['Crime_Count'] / 
                                             crime_per_capita_df['Total_Population'])

# Check the result
print(f"Shape: {crime_per_capita_df.shape}")
print(f"Columns: {crime_per_capita_df.columns.tolist()}")
print(f"\nNull values:\n{crime_per_capita_df.isnull().sum()}")

Shape: (304336, 20)
Columns: ['LSOA_Code', 'LSOA_Name', 'Year', 'Month', 'Month_Period', 'Crime_Count', 'Total_Population', 'LSOA_Latitude', 'LSOA_Longitude', 'LSOA_Shape_Area', 'LSOA_Shape_Length', 'MSOA_Code', 'MSOA_Name', 'Total_Annual_Income_British_Pounds', 'Max_Temperature_Celsius', 'Min_Temperature_Celsius', 'Rainfall_mm', 'Sunshine_Hours', 'Air_Frost_Days', 'Crime_per_Capita']

Null values:
LSOA_Code                                0
LSOA_Name                                0
Year                                     0
Month                                    0
Month_Period                             0
Crime_Count                              0
Total_Population                         0
LSOA_Latitude                            0
LSOA_Longitude                           0
LSOA_Shape_Area                          0
LSOA_Shape_Length                        0
MSOA_Code                                0
MSOA_Name                             4696
Total_Annual_Income_British_Pounds    4

In [64]:
# Fill missing MSOA_Name values with sequential labels: Missing Name 1, Missing Name 2, ...
missing_mask = crime_per_capita_df['MSOA_Name'].isna() | (crime_per_capita_df['MSOA_Name'].astype(str).str.strip() == "")
missing_count = missing_mask.sum()

crime_per_capita_df.loc[missing_mask, 'MSOA_Name'] = [f"Missing Name {i}" for i in range(1, missing_count + 1)]

print(f"Filled {missing_count} missing MSOA_Name values.")
print(f"Remaining missing MSOA_Name values: {crime_per_capita_df['MSOA_Name'].isna().sum()}")

Filled 4696 missing MSOA_Name values.
Remaining missing MSOA_Name values: 0


In [65]:
# Calculate the average income per year
yearly_avg_income = crime_per_capita_df.groupby('Year')['Total_Annual_Income_British_Pounds'].transform('mean')

# Fill missing values with the year-specific average
crime_per_capita_df['Total_Annual_Income_British_Pounds'] = crime_per_capita_df['Total_Annual_Income_British_Pounds'].fillna(yearly_avg_income)

# Verify the fill
print(f"Null values in Total_Annual_Income_British_Pounds after filling: {crime_per_capita_df['Total_Annual_Income_British_Pounds'].isnull().sum()}")
print(f"\nNull values by column:\n{crime_per_capita_df.isnull().sum()}")

Null values in Total_Annual_Income_British_Pounds after filling: 0

Null values by column:
LSOA_Code                             0
LSOA_Name                             0
Year                                  0
Month                                 0
Month_Period                          0
Crime_Count                           0
Total_Population                      0
LSOA_Latitude                         0
LSOA_Longitude                        0
LSOA_Shape_Area                       0
LSOA_Shape_Length                     0
MSOA_Code                             0
MSOA_Name                             0
Total_Annual_Income_British_Pounds    0
Max_Temperature_Celsius               0
Min_Temperature_Celsius               0
Rainfall_mm                           0
Sunshine_Hours                        0
Air_Frost_Days                        0
Crime_per_Capita                      0
dtype: int64


In [66]:
# Sort by LSOA_Code, Year, Month to ensure proper ordering for lag/rolling features
crime_per_capita_df = crime_per_capita_df.sort_values(['LSOA_Code', 'Year', 'Month']).reset_index(drop=True)

# ===== LAG FEATURES =====
# Create lags for each LSOA separately to avoid cross-boundary leakage
for lag in [1, 2, 3, 6]:
    crime_per_capita_df[f'crime_lag_{lag}m'] = crime_per_capita_df.groupby('LSOA_Code')['Crime_Count'].shift(lag)

# ===== ROLLING STATISTICS =====
# 3-month rolling mean and std
crime_per_capita_df['crime_rolling_3m_mean'] = crime_per_capita_df.groupby('LSOA_Code')['Crime_Count'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
crime_per_capita_df['crime_rolling_3m_std'] = crime_per_capita_df.groupby('LSOA_Code')['Crime_Count'].transform(
    lambda x: x.rolling(window=3, min_periods=1).std()
)

# 6-month rolling mean and std
crime_per_capita_df['crime_rolling_6m_mean'] = crime_per_capita_df.groupby('LSOA_Code')['Crime_Count'].transform(
    lambda x: x.rolling(window=6, min_periods=1).mean()
)
crime_per_capita_df['crime_rolling_6m_std'] = crime_per_capita_df.groupby('LSOA_Code')['Crime_Count'].transform(
    lambda x: x.rolling(window=6, min_periods=1).std()
)

# ===== POPULATION DENSITY =====
crime_per_capita_df['pop_density'] = (crime_per_capita_df['Total_Population'] / 
                                       crime_per_capita_df['LSOA_Shape_Area'])

# ===== INCOME PER CAPITA =====
crime_per_capita_df['income_per_capita'] = (crime_per_capita_df['Total_Annual_Income_British_Pounds'] / 
                                             crime_per_capita_df['Total_Population'])

# ===== MONTH SINUSOIDAL ENCODING =====
crime_per_capita_df['month_sin'] = np.sin(2 * np.pi * crime_per_capita_df['Month'] / 12)
crime_per_capita_df['month_cos'] = np.cos(2 * np.pi * crime_per_capita_df['Month'] / 12)

# ===== SEASON FLAGS =====
crime_per_capita_df['is_summer'] = crime_per_capita_df['Month'].isin([6, 7, 8]).astype(int)
crime_per_capita_df['is_winter'] = crime_per_capita_df['Month'].isin([12, 1, 2]).astype(int)
crime_per_capita_df['is_spring'] = crime_per_capita_df['Month'].isin([3, 4, 5]).astype(int)
crime_per_capita_df['is_autumn'] = crime_per_capita_df['Month'].isin([9, 10, 11]).astype(int)

# ===== MSOA AVERAGE CRIME =====
# Calculate average monthly crime count for each MSOA-Year-Month combination
msoa_avg = crime_per_capita_df.groupby(['MSOA_Code', 'Year', 'Month'])['Crime_Count'].transform('mean')
crime_per_capita_df['msoa_avg_crime'] = msoa_avg

# ===== CRIME RATE (normalized per 1000 people) =====
crime_per_capita_df['Crime_Rate'] = (crime_per_capita_df['Crime_Count'] / 
                                      crime_per_capita_df['Total_Population'] * 1000)

# ===== INCOME DEPRIVATION (normalized: 0=wealthy, 1=deprived) =====
max_income = crime_per_capita_df['income_per_capita'].max()
crime_per_capita_df['Income_Deprivation'] = (1 - 
    (crime_per_capita_df['income_per_capita'] / max_income))

# Check results
print(f"New shape: {crime_per_capita_df.shape}")
print(f"\nNew features created:")
new_features = [
    'crime_lag_1m', 'crime_lag_2m', 'crime_lag_3m', 'crime_lag_6m',
    'crime_rolling_3m_mean', 'crime_rolling_3m_std', 'crime_rolling_6m_mean', 'crime_rolling_6m_std',
    'Crime_Rate', 'pop_density', 'income_per_capita', 'Income_Deprivation',
    'month_sin', 'month_cos',
    'is_summer', 'is_winter', 'is_spring', 'is_autumn',
    'msoa_avg_crime'
]
print("\n".join(new_features))

print(f"\nNull values in new features:")
print(crime_per_capita_df[new_features].isnull().sum())

New shape: (304336, 39)

New features created:
crime_lag_1m
crime_lag_2m
crime_lag_3m
crime_lag_6m
crime_rolling_3m_mean
crime_rolling_3m_std
crime_rolling_6m_mean
crime_rolling_6m_std
Crime_Rate
pop_density
income_per_capita
Income_Deprivation
month_sin
month_cos
is_summer
is_winter
is_spring
is_autumn
msoa_avg_crime

Null values in new features:
crime_lag_1m             12415
crime_lag_2m             20053
crime_lag_3m             26393
crime_lag_6m             43206
crime_rolling_3m_mean        0
crime_rolling_3m_std     12415
crime_rolling_6m_mean        0
crime_rolling_6m_std     12415
Crime_Rate                   0
pop_density                  0
income_per_capita            0
Income_Deprivation           0
month_sin                    0
month_cos                    0
is_summer                    0
is_winter                    0
is_spring                    0
is_autumn                    0
msoa_avg_crime               0
dtype: int64


In [67]:
# Final dataframe with all required columns in logical order
crime_per_capita_df = crime_per_capita_df[[
    # === Identifiers and Names ===
    'LSOA_Code', 'LSOA_Name', 'MSOA_Code', 'MSOA_Name',
    
    # === Temporal Features ===
    'Year', 'Month', 'Month_Period',
    
    # === Target Variables ===
    'Crime_Count', 'Crime_per_Capita', 'Crime_Rate', 
    
    # === Population Features ===
    'Total_Population', 'pop_density',
    
    # === Income Features ===
    'Total_Annual_Income_British_Pounds', 'income_per_capita','Income_Deprivation',
    
    # === Geographic/Spatial Features ===
    'LSOA_Latitude', 'LSOA_Longitude', 'LSOA_Shape_Area', 'LSOA_Shape_Length',
    
    # === Weather Features ===
    'Max_Temperature_Celsius', 'Min_Temperature_Celsius',
    'Rainfall_mm', 'Sunshine_Hours', 'Air_Frost_Days',
    
    # === Temporal/Seasonal Encoding ===
    'month_sin', 'month_cos',
    'is_summer', 'is_winter', 'is_spring', 'is_autumn',
    
    # === Lag Features (Previous Months) ===
    'crime_lag_1m', 'crime_lag_2m', 'crime_lag_3m', 'crime_lag_6m',
    
    # === Rolling Statistics ===
    'crime_rolling_3m_mean', 'crime_rolling_3m_std',
    'crime_rolling_6m_mean', 'crime_rolling_6m_std',
    
    # === Aggregated Features ===
    'msoa_avg_crime'
]]

# Verify the reorganization
print(f"Final shape: {crime_per_capita_df.shape}")
print(f"\nFinal column order ({len(crime_per_capita_df.columns)} columns):")
for i, col in enumerate(crime_per_capita_df.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\nData types:")
print(crime_per_capita_df.dtypes)

print(f"\nFirst few rows:")
print(crime_per_capita_df.head())

Final shape: (304336, 39)

Final column order (39 columns):
 1. LSOA_Code
 2. LSOA_Name
 3. MSOA_Code
 4. MSOA_Name
 5. Year
 6. Month
 7. Month_Period
 8. Crime_Count
 9. Crime_per_Capita
10. Crime_Rate
11. Total_Population
12. pop_density
13. Total_Annual_Income_British_Pounds
14. income_per_capita
15. Income_Deprivation
16. LSOA_Latitude
17. LSOA_Longitude
18. LSOA_Shape_Area
19. LSOA_Shape_Length
20. Max_Temperature_Celsius
21. Min_Temperature_Celsius
22. Rainfall_mm
23. Sunshine_Hours
24. Air_Frost_Days
25. month_sin
26. month_cos
27. is_summer
28. is_winter
29. is_spring
30. is_autumn
31. crime_lag_1m
32. crime_lag_2m
33. crime_lag_3m
34. crime_lag_6m
35. crime_rolling_3m_mean
36. crime_rolling_3m_std
37. crime_rolling_6m_mean
38. crime_rolling_6m_std
39. msoa_avg_crime

Data types:
LSOA_Code                              object
LSOA_Name                              object
MSOA_Code                              object
MSOA_Name                              object
Year            

In [68]:
# Define lag and rolling columns
lag_cols = ['crime_lag_1m', 'crime_lag_2m', 'crime_lag_3m', 'crime_lag_6m']
rolling_cols = ['crime_rolling_3m_std', 'crime_rolling_6m_std']

# Strategy 1: Forward fill lag features within each LSOA (preserves temporal relationship)
for lag_col in lag_cols:
    crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col].fillna(method='ffill')

# Strategy 2: Fill remaining NaNs in lags with 0 (no crime history available)
crime_per_capita_df[lag_cols] = crime_per_capita_df[lag_cols].fillna(0)

# Strategy 3: Fill rolling std with 0 (no variation when insufficient data points)
crime_per_capita_df[rolling_cols] = crime_per_capita_df[rolling_cols].fillna(0)


# Final check for any remaining NaNs
remaining_nulls = crime_per_capita_df.isnull().sum()
if remaining_nulls.sum() == 0:
    print("All missing values handled successfully!")
else:
    print("Remaining missing values:")
    print(remaining_nulls[remaining_nulls > 0])

print(f"\nFinal shape: {crime_per_capita_df.shape}")
print(f"Year range: {crime_per_capita_df['Year'].min()} - {crime_per_capita_df['Year'].max()}")
print(f"Total records: {len(crime_per_capita_df):,}")

/tmp/ipykernel_94916/2422914378.py:7: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col].fillna(method='ffill')
/tmp/ipykernel_94916/2422914378.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col].fillna(method='ffill')


/tmp/ipykernel_94916/2422914378.py:7: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col].fillna(method='ffill')
/tmp/ipykernel_94916/2422914378.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col].fillna(method='ffill')
/tmp/ipykernel_94916/2422914378.py:7: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  crime_per_capita_df[lag_col] = crime_per_capita_df.groupby('LSOA_Code')[lag_col

All missing values handled successfully!

Final shape: (304336, 39)
Year range: 2020 - 2024
Total records: 304,336


In [69]:
crime_per_capita_df.isnull().sum()

LSOA_Code                             0
LSOA_Name                             0
MSOA_Code                             0
MSOA_Name                             0
Year                                  0
Month                                 0
Month_Period                          0
Crime_Count                           0
Crime_per_Capita                      0
Crime_Rate                            0
Total_Population                      0
pop_density                           0
Total_Annual_Income_British_Pounds    0
income_per_capita                     0
Income_Deprivation                    0
LSOA_Latitude                         0
LSOA_Longitude                        0
LSOA_Shape_Area                       0
LSOA_Shape_Length                     0
Max_Temperature_Celsius               0
Min_Temperature_Celsius               0
Rainfall_mm                           0
Sunshine_Hours                        0
Air_Frost_Days                        0
month_sin                             0


In [70]:
import os

# Create the directory if it doesn't exist
output_dir = 'Datasets/Data-For-Models/Crime-Per-Capita-Dataset'
os.makedirs(output_dir, exist_ok=True)

# Export the dataframe to CSV
crime_per_capita_df.to_csv(f'{output_dir}/crime_per_capita_df.csv', index=False)
print(f"crime_per_capita_df exported successfully to '{output_dir}/crime_per_capita_df.csv'")
print(f"File size: {os.path.getsize(f'{output_dir}/crime_per_capita_df.csv') / (1024**2):.2f} MB")

crime_per_capita_df exported successfully to 'Datasets/Data-For-Models/Crime-Per-Capita-Dataset/crime_per_capita_df.csv'
File size: 110.50 MB


In [71]:
crime_rate = pd.read_csv("Datasets/Data-For-Models/Crime-Per-Capita-Dataset/crime_per_capita_df.csv")


In [72]:
crime_rate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304336 entries, 0 to 304335
Data columns (total 39 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   LSOA_Code                           304336 non-null  object 
 1   LSOA_Name                           304336 non-null  object 
 2   MSOA_Code                           304336 non-null  object 
 3   MSOA_Name                           304336 non-null  object 
 4   Year                                304336 non-null  int64  
 5   Month                               304336 non-null  int64  
 6   Month_Period                        304336 non-null  object 
 7   Crime_Count                         304336 non-null  int64  
 8   Crime_per_Capita                    304336 non-null  float64
 9   Crime_Rate                          304336 non-null  float64
 10  Total_Population                    304336 non-null  int64  
 11  pop_density               

## Task 2 - Dataset for Crime Hotspot Prediction